# Базы данных (Воловиков) — занятие 1
### ACID, группы команд SQL и домашняя работа

**Логика занятия:** свойства транзакций ACID → группы команд SQL → разбор схемы и запроса из домашней работы на реальном примере.

## 1. ACID — свойства транзакций

1. **Атомарность** — транзакция выполняется целиком или не выполняется вовсе.
2. **Согласованность** — данные переходят из одного корректного состояния в другое.
3. **Изолированность** — параллельные транзакции не влияют друг на друга (меньше всего подходит к SQLite).
4. **Устойчивость** — после сбоя данные сохраняются.

**Сериализация** нужна, чтобы упорядочить одновременные действия разных транзакций.

## 2. Группы команд SQL

| Группа | Назначение |
| --- | --- |
| **DDL** | структура базы данных (`CREATE`, `ALTER`, `DROP`) |
| **DML** | работа с данными (`SELECT`, `INSERT`, `UPDATE`, `DELETE`) |
| **DCL** | права доступа (`GRANT`, `REVOKE`) |
| **TCL** | работа с транзакциями (`COMMIT`, `ROLLBACK`) |

## 3. Домашняя работа: схема

Пять таблиц: `positions`, `eq_types`, `users`, `packs`, `parametrs`. Создадим их в SQLite и заполним тестовыми данными.

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect(':memory:')
cur = con.cursor()
cur.executescript('''
drop table if exists parametrs;
drop table if exists packs;
drop table if exists users;
drop table if exists eq_types;
drop table if exists positions;

create table positions(
  id integer primary key,
  name varchar(150) not null unique
);
create table eq_types(
  id integer primary key,
  name varchar(150) not null unique
);
create table users(
  id integer primary key,
  full_name varchar(150) not null,
  position_id integer not null references positions(id)
);
create table packs(
  id integer primary key,
  user_id integer not null references users(id),
  eq_type_id integer not null references eq_types(id),
  created_on date not null
);
create table parametrs(
  id integer primary key,
  pack_id integer not null references packs(id),
  name varchar(100) not null,
  unit varchar(20),
  value numeric(10,2) not null
);
''')
print('таблицы созданы:', cur.execute(
    "select name from sqlite_master where type='table' order by name"
).fetchall())

таблицы созданы: [('eq_types',), ('packs',), ('parametrs',), ('positions',), ('users',)]


In [2]:
cur.executescript('''
insert into eq_types (id, name) values (1, 'ДМК'), (2, 'ВР');
insert into positions (id, name) values
  (1, 'оператор метеопоста'),
  (2, 'начальник метеопоста'),
  (3, 'техник');
insert into users (id, full_name, position_id) values
  (1, 'Иванов Иван', 1),
  (2, 'Петров Пётр', 2);
insert into packs (id, user_id, eq_type_id, created_on) values
  (1, 1, 1, '2026-09-01'),
  (2, 2, 2, '2026-09-02');
insert into parametrs (id, pack_id, name, unit, value) values
  (1, 1, 'высота метеопоста', 'м', 100),
  (2, 1, 'температура воздуха на высоте метеопоста', '*С', 15),
  (3, 1, 'давление атмосферы на высоте метеопоста', 'мм рт. ст.', 750),
  (4, 1, 'направление ветра в приземном слое воздуха', 'б. д. у.', 0),
  (5, 1, 'скорость ветра в приземном слое воздуха', 'м/с', 0),
  (6, 2, 'высота метеопоста', 'м', 100),
  (7, 2, 'температура воздуха на высоте метеопоста', '*С', 12),
  (8, 2, 'давление атмосферы на высоте метеопоста', 'мм рт. ст.', 760),
  (9, 2, 'направление ветра в приземном слое воздуха', 'б. д. у.', 20),
  (10, 2, 'дальность сноса пуль', 'м', 45);
''')
con.commit()
print('данные добавлены')

данные добавлены


## 4. Соединение таблиц (JOIN)

Итоговый запрос домашней работы собирает данные из всех таблиц:

In [3]:
query = '''
select p.id           as pack_id,
       p.created_on,
       u.full_name,
       pos.name       as position,
       et.name        as eq_type,
       par.name       as parameter,
       par.unit,
       par.value
from packs p
join users u        on p.user_id = u.id
join positions pos  on u.position_id = pos.id
join eq_types et    on p.eq_type_id = et.id
join parametrs par  on par.pack_id = p.id
order by p.id, par.id;
'''
df = pd.read_sql_query(query, con)
print('строк в результате:', len(df))
df.head(10)

строк в результате: 10


,pack_id,created_on,full_name,position,eq_type,parameter,unit,value
0,1,2026-09-01,Иванов Иван,оператор метеопоста,ДМК,высота метеопоста,м,100
1,1,2026-09-01,Иванов Иван,оператор метеопоста,ДМК,температура воздуха на высоте метеопоста,*С,15
2,1,2026-09-01,Иванов Иван,оператор метеопоста,ДМК,давление атмосферы на высоте метеопоста,мм рт. ст.,750
3,1,2026-09-01,Иванов Иван,оператор метеопоста,ДМК,направление ветра в приземном слое воздуха,б. д. у.,0
4,1,2026-09-01,Иванов Иван,оператор метеопоста,ДМК,скорость ветра в приземном слое воздуха,м/с,0
5,2,2026-09-02,Петров Пётр,начальник метеопоста,ВР,высота метеопоста,м,100
6,2,2026-09-02,Петров Пётр,начальник метеопоста,ВР,температура воздуха на высоте метеопоста,*С,12
7,2,2026-09-02,Петров Пётр,начальник метеопоста,ВР,давление атмосферы на высоте метеопоста,мм рт. ст.,760
8,2,2026-09-02,Петров Пётр,начальник метеопоста,ВР,направление ветра в приземном слое воздуха,б. д. у.,20
9,2,2026-09-02,Петров Пётр,начальник метеопоста,ВР,дальность сноса пуль,м,45


## 5. Транзакции и ACID на практике

Покажем атомарность: `ROLLBACK` отменяет изменения целиком.

In [4]:
cur.execute("insert into eq_types (id, name) values (3, 'ТЕСТ')")
print('до отката:', cur.execute('select count(*) from eq_types').fetchone()[0])
con.rollback()
print('после отката:', cur.execute('select count(*) from eq_types').fetchone()[0])
con.close()

до отката: 3
после отката: 2


---

**Итог.** Разобрали свойства ACID и сериализацию, группы команд SQL (DDL, DML, DCL, TCL) и собрали запрос с соединением пяти таблиц из домашней работы.

Три формы нормализации 

1. в рамках одного значения все уникально
2. в рамках одной строки
3. 